# 04 — Sequence: multi-image time series

A `Sequence` strings together many image pairs and solves them efficiently,
re-using mesh geometry and warp guesses between frames.

This notebook covers: `Sequence`, `SequenceOptions` (guide / sequential / sync),
per-pair data access, and an animated displacement history.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import geopyv_dev as gp

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "animation.embed_limit": 50,
})


In [ ]:
# Generate a 5-image shear-band sequence (images seq_0 … seq_4).
os.makedirs("data/sequence", exist_ok=True)
speckle_seq = gp.Speckle(
    image_cfg={"image_dir": "data/sequence", "name": "seq", "image_size": (401, 401)},
    speckle_cfg={"speckle_size": 3.5, "speckle_number": 700},
    progression="deformation",
    deformation_cfg={
        "comp": [8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        "mode": "SB",
        "option": "sin",
        "width": 100.0,
    },
    noise_cfg=(0.2, 1.5),
    scale_cfg={"scale": "lin", "n": 5},
)
speckle_seq.solve(seed=42)
print("Generated: data/sequence/seq_0.jpg … seq_4.jpg")

# Mesh configuration shared across all pairs
boundary_nodes = np.array([
    [50.0, 50.0], [50.0, 350.0], [350.0, 350.0], [350.0, 50.0]
])
boundary  = gp.PathRegion(nodes=boundary_nodes, hard=False)
local_mask = gp.Mask(mask_type="local", shape="circle", size=25)
seed_coord = [200.0, 200.0]


## `SequenceOptions` — how pairs connect

`SequenceOptions` controls how information flows from one pair to the next:

| Option | Default | Effect |
|--------|---------|--------|
| `guide` | `True` | Use the previous pair's warp field to seed the next pair's solve |
| `sequential` | `False` | Advance the reference image after each successful pair |
| `sync` | `True` | Re-use the previous mesh geometry (skip CDT) for subsequent pairs |
| `override_` | `False` | Relax tolerance on retry after a failed consecutive pair |

`sync=True` is almost always what you want — it saves time and keeps the node
numbering consistent across pairs so you can directly index `sequence.displacements(i)`.


In [ ]:
# Show SequenceOptions defaults
opts = gp.SequenceOptions()
print(opts)

# For a standard analysis: guide warp + sync geometry
opts_standard = gp.SequenceOptions(guide=True, sync=True, sequential=False)
print(opts_standard)


In [ ]:
# Build the Sequence (discovers all images in data/sequence/ automatically)
sequence = gp.Sequence(
    image_dir   = "data/sequence",
    boundary    = boundary,
    target_nodes= 60,
    size        = (15., 70.),
    mesh_order  = 1,
)
print(f"{sequence}")
print(f"  Image paths found: {sequence.n_pairs + 1}")
for p in sequence.image_paths[:3]:
    print(f"    {os.path.basename(p)}")
print("    …")


In [ ]:
# Solve all pairs  (a few seconds per pair)
sequence.solve(
    local_mask    = local_mask,
    seed_coord    = seed_coord,
    seed_warp     = [0.0] * 6,
    subset_order  = 1,
    tolerance     = 0.75,
    seed_tolerance= 0.9,
    method        = "icgn",
    options       = gp.SequenceOptions(guide=True, sync=True),
)
print(sequence)
print(f"  solved     : {sequence.solved}")
print(f"  n_pairs    : {sequence.n_pairs}")
print(f"  pair 0 mean C_ZNCC : {sequence.c_zncc(0).mean():.4f}")


In [ ]:
# Access per-pair data without loading full Mesh objects
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, ax in enumerate(axes.flat):
    nodes = sequence.nodes(idx)
    u     = sequence.displacements(idx)[:, 0]   # x-displacements
    zncc  = sequence.c_zncc(idx)
    # Quick per-node scatter coloured by u
    sc = ax.scatter(nodes[:, 0], nodes[:, 1], c=u, cmap="RdBu_r",
                    s=35, linewidths=0)
    plt.colorbar(sc, ax=ax, fraction=0.046, label="u (px)")
    ax.set_title(f"Pair {idx}: u-displacement", fontsize=11, fontweight="bold")
    ax.set_aspect("equal"); ax.set_xlim(0, 401); ax.set_ylim(0, 401)
    ax.invert_yaxis(); ax.axis("off")

plt.suptitle("u-displacement for each image pair", fontsize=13, fontweight="bold")
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
fig.savefig("../docs/assets/nb4_displacement_grid.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Animated u-displacement across pairs
nodes = sequence.nodes(0)
all_u = [sequence.displacements(i)[:, 0] for i in range(sequence.n_pairs)]
vmin, vmax = min(u.min() for u in all_u), max(u.max() for u in all_u)

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(nodes[:, 0], nodes[:, 1], c=all_u[0], cmap="RdBu_r",
                s=40, linewidths=0, vmin=vmin, vmax=vmax)
cb = plt.colorbar(sc, ax=ax, fraction=0.046, label="u-displacement (px)")
title = ax.set_title("Pair 0: u-displacement", fontsize=11, fontweight="bold")
ax.set_aspect("equal"); ax.set_xlim(0, 401); ax.set_ylim(0, 401)
ax.invert_yaxis(); ax.axis("off")

def update_seq(frame):
    sc.set_array(all_u[frame])
    title.set_text(f"Pair {frame}: u-displacement  (mean={all_u[frame].mean():.3f} px)")
    return sc, title

anim_seq = FuncAnimation(fig, update_seq, frames=sequence.n_pairs, interval=600, blit=True)
plt.tight_layout()
anim_seq.save("../docs/assets/nb4_sequence_animation.gif", writer="pillow", fps=2)
HTML(anim_seq.to_jshtml())


In [ ]:
# Save and reload
sequence.save("data/sequence.pyv")
seq_loaded = gp.load("data/sequence.pyv")
print(f"Loaded: {seq_loaded}")
print(f"  solved  : {seq_loaded.solved}")
print(f"  n_pairs : {seq_loaded.n_pairs}")
print(f"  pair 0 mean C_ZNCC : {seq_loaded.c_zncc(0).mean():.4f}")


## Key takeaways

- `Sequence` auto-discovers images from a directory by trailing integer sort.
- `SequenceOptions(guide=True, sync=True)` is the recommended default.
- Access per-pair arrays directly via `sequence.nodes(i)`, `sequence.displacements(i)`,
  `sequence.c_zncc(i)` — faster than `sequence.mesh_solutions[i]` for large sequences.
- The `.pyv` format stores all pairs compactly; reload with `gp.load()`.

**Next:** Notebook 05 — tracking material points through time with `Particle` and `Field`.
